# Memory

<div style="display: flex; justify-content: flex-start; gap: 10px;">
  <img src="./assets/LC_Memory_before.png" style="width:500px; border:1px solid #ccc; border-radius:6px;">
</div>

Persisting messages, or 'agent state' between invocations of the agent.

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

OPENAI_API_KEY=****32d0
LANGSMITH_API_KEY=****c830
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


In [2]:
from sql_db import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

In [3]:
from dataclasses import dataclass

@dataclass
class RuntimeContext:
    db: SQLDatabase

In [4]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [5]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:minimax/minimax-m3:free",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

## Repeated Queries

In [7]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
    context=RuntimeContext(db=db),
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================

I'll help you find that information. Let me first check if there's a database available with customer and invoice data.
Tool Calls:
  execute_sql (call_01a073615c177c0392fa09c6)
 Call ID: call_01a073615c177c0392fa09c6
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' LIMIT 20
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_01a0736163147dd18466c974)
 Call ID: call_01a0736163147dd18466c974
  Args:
    query: SELECT CustomerId,

In [8]:
question = "What were the titles?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================

I'd be happy to help, but I need more context. Could you clarify what you're asking about? For example:

- Are you asking about movie titles, book titles, job titles, etc.?
- Do you want me to query a specific database or table?

If you have a SQLite database available, please let me know which table contains the titles you're interested in, and I'll run a query for you.


## Add memory

In [9]:
from langgraph.checkpoint.memory import InMemorySaver

In [10]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

agent = create_agent(
    model="openai:minimax/minimax-m3:free",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=InMemorySaver(),
)

In [11]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================

I'll help you find the total on your last invoice, Frank. Let me first check the database structure to understand what we're working with.
Tool Calls:
  execute_sql (call_01a07362287e7ce083c239e1)
 Call ID: call_01a07362287e7ce083c239e1
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table'
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_01a073622dbd7d20933a7914)
 Call ID: call_01a073622dbd7d20933a7914
  Args:
    query: SELECT F

In [12]:
question = "What were the titles?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================

Let me find the tracks on your last invoice.
Tool Calls:
  execute_sql (call_01a073625496775183ea696f)
 Call ID: call_01a073625496775183ea696f
  Args:
    query: SELECT t.Name FROM Track t JOIN InvoiceLine il ON t.TrackId = il.TrackId WHERE il.InvoiceId = 374
================================= Tool Message =================================
Name: execute_sql

[('Holier Than Thou',), ('Through The Never',), ('My Friend Of Misery',), ('The Wait',), ('Blitzkrieg',), ('So What',)]
================================== Ai Message ==================================

The tracks on your last invoice were:

1. Holier Than Thou
2. Through The Never
3. My Friend Of Misery
4. The Wait
5. Blitzkrieg
6. So What

Looks like a nice selection! Anything else you'd like to know?


## Try your own queries
Now that there is memory, check the agents recall!

In [13]:
question = "What is my most costliest purchase?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

What is my most costliest purchase?
================================== Ai Message ==================================

Let me find your most expensive invoice, Frank.
Tool Calls:
  execute_sql (call_01a07362bec87672ad20fd9f)
 Call ID: call_01a07362bec87672ad20fd9f
  Args:
    query: SELECT InvoiceId, InvoiceDate, Total FROM Invoice WHERE CustomerId = (SELECT CustomerId FROM Customer WHERE FirstName = 'Frank' AND LastName = 'Harris') ORDER BY Total DESC LIMIT 1
================================= Tool Message =================================
Name: execute_sql

[(145, '2010-09-23 00:00:00', 13.86)]
================================== Ai Message ==================================

Your most expensive purchase was:

- **Invoice ID:** 145
- **Invoice Date:** September 23, 2010
- **Total:** $13.86

Would you like me to look up the tracks on that invoice as well?
